# 정수기 구매·이용 통합 상담 AI: 데이터 분석

분석 기준일 **2026-09-29**. 프로젝트 루트 또는 이 노트북 폴더에서 실행할 수 있다. 필요한 패키지는 `pandas`, `Pillow`다.

**질문:** 구매·설치, 기능, 사용·오류, 관리·구독, 계약·해지에 걸친 상담 근거가 있는가? 이 공개 자료만으로 어디까지 말할 수 있는가?

**자료 구분:** 1372 품목별 상담은 접수 규모, KCA 피해구제는 피해 신청 사유, 표준답변과 LG FAQ는 게시 문서 구성, 제품 상세는 답변에 쓸 수 있는 사양이다. 모집단이 달라 서로 합치거나 같은 비율로 해석하지 않는다.


In [1]:
from pathlib import Path
import json
import pandas as pd

def find_project_root(start):
    for path in (start, *start.parents):
        if (path / "핵심프로젝트.md").exists():
            return path
    raise FileNotFoundError("핵심프로젝트.md가 있는 프로젝트 폴더에서 실행하세요.")

ROOT = find_project_root(Path.cwd())
DATA = ROOT / "01_제출용/01_데이터분석/자료"
NOTEBOOK_DIR = ROOT / "01_제출용" / "01_데이터분석" / "노트북"

def read_korean_csv(path):
    for encoding in ("utf-8-sig", "cp949"):
        try:
            frame = pd.read_csv(path, encoding=encoding, dtype=str)
            frame.columns = frame.columns.str.strip()
            return frame.map(lambda value: value.strip() if isinstance(value, str) else value)
        except UnicodeDecodeError:
            pass
    raise ValueError(f"인코딩을 확인하세요: {path}")

print("프로젝트:", ROOT.name)
print("분석 대상 기간: 2025-07~2026-07")


프로젝트: 핵심프로젝트
분석 대상 기간: 2025-07~2026-07


## 1. 품목별 상담: 파일 품질과 규모

[한국소비자원 품목별 상담 현황](https://www.data.go.kr/data/15120072/fileData.do)의 월별 상위 500품목 파일 13개를 분석한다. 같은 고객의 중복 상담 여부는 알 수 없다.


In [2]:
monthly_paths = sorted((DATA / "원본/품목별상담_월별").glob("품목별_상담현황_*.csv"))
assert len(monthly_paths) == 13, f"기대한 13개 대신 {len(monthly_paths)}개"
monthly_frames = []
for path in monthly_paths:
    frame = read_korean_csv(path)
    assert len(frame) == 500
    assert set(["연도", "월", "품목명", "상담건수", "전월대비 증감률", "전년동월대비 증감률"]).issubset(frame.columns)
    frame["연월"] = frame["연도"] + "-" + frame["월"].str.zfill(2)
    assert frame["연월"].nunique() == 1
    assert frame["연월"].iloc[0].replace("-", "") == path.stem.rsplit("_", 1)[1][:6]
    frame["상담건수"] = pd.to_numeric(frame["상담건수"], errors="raise")
    monthly_frames.append(frame)
monthly_all = pd.concat(monthly_frames, ignore_index=True)
core = ["연도", "월", "품목명", "상담건수", "전월대비 증감률", "전년동월대비 증감률"]
assert not monthly_all[core].isna().any().any()
assert not monthly_all[[c for c in core if c != "상담건수"]].eq("").any().any()
assert not monthly_all.duplicated(["연월", "품목명"]).any()
assert (monthly_all["상담건수"] >= 0).all() and (monthly_all["상담건수"] % 1 == 0).all()
print(f"파일 {len(monthly_paths)}개 / 총 {len(monthly_all):,}행 / 핵심열 결측·공백 0 / 연월×품목 중복 0")
print("상담건수 음수·소수·변환 실패 0")


파일 13개 / 총 6,500행 / 핵심열 결측·공백 0 / 연월×품목 중복 0
상담건수 음수·소수·변환 실패 0


In [3]:
names = ["정수기대여(렌트)", "정수기", "냉장고", "TV"]
trend = monthly_all[monthly_all["품목명"].isin(names)].pivot(index="연월", columns="품목명", values="상담건수").sort_index()
assert len(trend) == 13 and trend.notna().all().all()
print("13개월 품목별 상담건수 합계")
print(trend.sum().reindex(names).to_string())
rental = trend["정수기대여(렌트)"]
print(f"정수기대여 월평균 {rental.mean():.1f}건, 범위 {rental.min()}~{rental.max()}건")
print(f"냉장고보다 많은 달 {(rental > trend['냉장고']).sum()}/13, TV보다 많은 달 {(rental > trend['TV']).sum()}/13")
print("\n월별 정수기대여 추이")
monthly_table = pd.DataFrame({"상담건수": rental, "전월대비_재계산_%": (rental.pct_change() * 100).round(1)})
print(monthly_table.to_string())


13개월 품목별 상담건수 합계
품목명
정수기대여(렌트)    6902
정수기          1098
냉장고          4775
TV           3811
정수기대여 월평균 530.9건, 범위 408~626건
냉장고보다 많은 달 13/13, TV보다 많은 달 13/13

월별 정수기대여 추이
         상담건수  전월대비_재계산_%
연월                       
2025-07   626         NaN
2025-08   569        -9.1
2025-09   564        -0.9
2025-10   431       -23.6
2025-11   494        14.6
2025-12   503         1.8
2026-01   479        -4.8
2026-02   408       -14.8
2026-03   568        39.2
2026-04   572         0.7
2026-05   506       -11.5
2026-06   582        15.0
2026-07   600         3.1


In [4]:
july = monthly_all[(monthly_all["연월"] == "2026-07") & (monthly_all["품목명"] == "정수기대여(렌트)")].iloc[0]
recalc_mom = (rental.loc["2026-07"] / rental.loc["2026-06"] - 1) * 100
recalc_yoy = (rental.loc["2026-07"] / rental.loc["2025-07"] - 1) * 100
print(f"2026-07 제공 전월 증감률: {july['전월대비 증감률']} / 보유 월별 건수 재계산: {recalc_mom:+.1f}%")
print(f"2026-07 제공 전년동월 증감률: {july['전년동월대비 증감률']} / 보유 월별 건수 재계산: {recalc_yoy:+.1f}%")
print("불일치 이유는 원자료에 없음. 아래 판단에는 재계산값을 사용한다.")


2026-07 제공 전월 증감률: 3.8 / 보유 월별 건수 재계산: +3.1%
2026-07 제공 전년동월 증감률: -4.3 / 보유 월별 건수 재계산: -4.2%
불일치 이유는 원자료에 없음. 아래 판단에는 재계산값을 사용한다.


## 2. 정수기대여 표준답변과 KCA 피해구제

[소비자상담 표준답변](https://www.data.go.kr/data/15144809/fileData.do)의 `정수기대여` 43행은 **문서 항목**이다. [KCA 피해예방주의보](https://www.kca.go.kr/home/sub.do?menukey=4005&mode=view&no=1003878293)의 1,462건은 별도 **피해구제 신청**이다. 둘 다 1372 월별 접수건수와 합산하지 않는다.


In [5]:
qa = read_korean_csv(DATA / "원본/표준답변/소비자상담_표준답변_20250725.csv")
qa_rental = qa[qa["품목명"].eq("정수기대여")].copy()
qa_rental["상위구분"] = qa_rental["구분"].str.split("_").str[0]
unique_questions = qa_rental["질문"].str.replace(r"\s+", "", regex=True).nunique()
assert len(qa) == 1332 and len(qa_rental) == 43 and unique_questions == 31
print(f"표준답변 전체 {len(qa):,}행 / 정수기대여 {len(qa_rental)}행 / 공백 제거 후 고유 질문 {unique_questions}개")
print(qa_rental["상위구분"].value_counts().to_string())
print("※ 분류별 행 수는 상담 문의 빈도가 아니다.")


표준답변 전체 1,332행 / 정수기대여 43행 / 공백 제거 후 고유 질문 31개
상위구분
계약해지·해제      12
가격·요금        11
계약불이행        10
품질            5
안전(제품/시설)     2
계약해제·해지       1
청약철회          1
표시·광고         1
※ 분류별 행 수는 상담 문의 빈도가 아니다.


In [6]:
harm = json.loads((DATA / "가공/KCA_정수기_피해구제_요약_20250530.json").read_text(encoding="utf-8"))
harm_table = pd.DataFrame(harm["reasons"])
assert harm_table["cases"].sum() == harm["total_cases"] == 1462
assert sum(item["cases"] for item in harm["contract_detail"]) == 823
harm_table["비율_%"] = (harm_table["cases"] / 1462 * 100).round(1)
print("KCA 정수기 렌탈 피해구제 2022~2025-03 (분모 1,462건)")
print(harm_table.to_string(index=False))
print("계약 중 해지·위약금 503건, 계약불이행 277건")
print("※ 이 표는 피해구제 신청 사유이며 1372 상담이나 LG 고객의 사유별 비율이 아니다.")


KCA 정수기 렌탈 피해구제 2022~2025-03 (분모 1,462건)
category  cases  비율_%
      계약    823  56.3
   품질·AS    471  32.2
    부당행위     96   6.6
   가격·요금     21   1.4
      안전     19   1.3
   표시·약관     17   1.2
      기타     15   1.0
계약 중 해지·위약금 503건, 계약불이행 277건
※ 이 표는 피해구제 신청 사유이며 1372 상담이나 LG 고객의 사유별 비율이 아니다.


## 3. 추가 수집: LG 공식 FAQ와 제품 비교

[LG 공식 정수기 FAQ](https://www.lge.co.kr/story/faq/water-purifiers-faq)의 ‘정수기 제품 질문’ 20개를 화면 순서대로 수작업 분류했다. 아래 분포는 **게시 문서 구성**이다. 질문이 고객에게 얼마나 자주 발생했는지는 알 수 없다. [구매가이드](https://www.lge.co.kr/story/user-guide/lg-water-purifiers-guide)는 사용 목적·설치 공간과 살균·기능·설치/디자인을 선택 기준으로 제시한다.


In [7]:
faq = json.loads((DATA / "가공/LG_정수기_FAQ_질문목록_20260929.json").read_text(encoding="utf-8"))
faq_items = pd.DataFrame(faq["questions"])
assert len(faq_items) == 20 and faq_items["no"].tolist() == list(range(1, 21))
print("LG 공식 FAQ '정수기 제품 질문' 20개: 수작업 주제 분류")
print(faq_items["category"].value_counts().to_string())
print("※ 답변 범위 확인용. 실제 문의 빈도·불편 심각도·제품 추천 수요가 아니다.")


LG 공식 FAQ '정수기 제품 질문' 20개: 수작업 주제 분류
category
관리       5
사용·증상    5
구매·설치    4
기능       4
구독·계약    2
※ 답변 범위 확인용. 실제 문의 빈도·불편 심각도·제품 추천 수요가 아니다.


In [8]:
catalog = json.loads((DATA / "가공/LG_정수기_모델비교_20260929.json").read_text(encoding="utf-8"))
models = pd.DataFrame(catalog["model_comparison"])
assert set(models["model_id"]) == {"WD520VCT", "WU923AWB"}
comparison = models[["model_id", "type", "water_modes", "body_size_w_h_d_mm", "portion_ml", "cleaning_water", "source_url"]].copy()
for col in ["water_modes", "body_size_w_h_d_mm", "portion_ml"]:
    comparison[col] = comparison[col].map(lambda values: "/".join(str(v) for v in values))
print("추천 시연 후보 2모델의 공식 상세 스펙 (2026-09-29 확인)")
print(comparison.to_string(index=False))
print("※ 두 모델 비교는 조건 확인형 추천의 실행 가능성만 보여준다. 전 제품군에서의 최적 추천이나 구매 수요를 입증하지 않는다.")


추천 시연 후보 2모델의 공식 상세 스펙 (2026-09-29 확인)
model_id  type water_modes body_size_w_h_d_mm       portion_ml                 cleaning_water                                                            source_url
WD520VCT  데스크형    냉수/온수/정수        168/398/400 120/250/500/1000    공식 상세 페이지 스펙에 해당 항목 확인되지 않음 https://www.lge.co.kr/product/care-solutions/water-purifiers/wd520vct
WU923AWB 언더싱크형    냉수/온수/정수        170/420/410      120/250/500 공식 상세 페이지 스펙에 클린세척수(전해수) O로 표시 https://www.lge.co.kr/product/care-solutions/water-purifiers/wu923awb
※ 두 모델 비교는 조건 확인형 추천의 실행 가능성만 보여준다. 전 제품군에서의 최적 추천이나 구매 수요를 입증하지 않는다.


In [9]:
age = read_korean_csv(DATA / "원본/연령대별/연령대별_상담다발품목_20260731.csv")
match = age[age["품목명"].eq("정수기대여(렌트)")]
print("2026-07 연령별 상위5품목 파일의 정수기대여 항목")
print(match[["연령대", "순위", "상담건수", "상담비율"]].to_string(index=False))
print("※ 다른 연령대에 상위5 항목이 없다는 사실은 해당 연령 상담 0건을 뜻하지 않는다.")


2026-07 연령별 상위5품목 파일의 정수기대여 항목
연령대 순위 상담건수 상담비율
60대  5   87  1.5
※ 다른 연령대에 상위5 항목이 없다는 사실은 해당 연령 상담 0건을 뜻하지 않는다.


## 4. 그래프 재생성과 해석 경계

다음 셀은 이 폴더의 `분석_재현.py`를 사용해 두 PNG를 원본에서 다시 만든다.


In [10]:
import importlib.util
spec = importlib.util.spec_from_file_location("purifier_trend", NOTEBOOK_DIR / "분석_재현.py")
chart_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(chart_module)
chart_module.draw_trend(chart_module.monthly_data())
chart_module.draw_categories(chart_module.read_csv(chart_module.QA))
print("그래프 재생성 완료:")
print(NOTEBOOK_DIR.parent / "그래프" / "그래프_13개월_상담추이.png")
print(NOTEBOOK_DIR.parent / "그래프" / "그래프_표준답변_문서구성.png")


그래프 재생성 완료:
/Users/kimhyunjin/Library/CloudStorage/OneDrive-에이아이캠퍼스/바탕 화면/LGAxSchool/핵심프로젝트/정수기프로젝트/01_데이터분석/그래프/그래프_13개월_상담추이.png
/Users/kimhyunjin/Library/CloudStorage/OneDrive-에이아이캠퍼스/바탕 화면/LGAxSchool/핵심프로젝트/정수기프로젝트/01_데이터분석/그래프/그래프_표준답변_문서구성.png


In [11]:
from PIL import Image, ImageDraw, ImageFont
font_path = "/System/Library/Fonts/AppleSDGothicNeo.ttc"
font = lambda size: ImageFont.truetype(font_path, size)
canvas = Image.new("RGB", (1280, 690), "#ffffff")
draw = ImageDraw.Draw(canvas)
draw.text((60, 30), "정수기 통합 상담 근거: 서로 다른 두 자료", font=font(34), fill="#17212f")
draw.text((60, 82), "왼쪽은 피해구제 신청 사유, 오른쪽은 LG 공식 FAQ 문서 주제입니다. 서로 합산하지 않습니다.", font=font(19), fill="#5c6675")

def panel(x, title, rows, maximum, subtitle):
    draw.rounded_rectangle((x, 138, x + 560, 625), radius=20, fill="#f6f8fa")
    draw.text((x + 28, 165), title, font=font(25), fill="#17212f")
    draw.text((x + 28, 205), subtitle, font=font(17), fill="#64748b")
    for i, (label, value) in enumerate(rows):
        y = 257 + i * 67
        draw.text((x + 28, y), label, font=font(18), fill="#253244")
        draw.rounded_rectangle((x + 180, y + 2, x + 180 + 300 * value / maximum, y + 27), radius=5, fill="#b32045")
        draw.text((x + 493, y), str(value), font=font(18), fill="#253244")

left = [("계약", 823), ("품질·AS", 471), ("그 외", 168)]
right = [(label, int(value)) for label, value in faq_items["category"].value_counts().items()]
panel(50, "KCA 피해구제: 1,462건", left, 823, "2022년~2025년 3월 / 피해 신청 건수")
panel(670, "LG 공식 FAQ: 20항목", right, 5, "2026-09-29 확인 / 게시 문서 수")
draw.text((60, 654), "출처: KCA 정수기 렌탈 피해예방주의보 · LG 정수기 고객 Q&A / 비율·빈도 직접 비교 금지", font=font(17), fill="#64748b")
chart_path = NOTEBOOK_DIR.parent / "그래프" / "그래프_통합상담_근거범위.png"
canvas.save(chart_path)
print("추가 그래프:", chart_path)


추가 그래프: /Users/kimhyunjin/Library/CloudStorage/OneDrive-에이아이캠퍼스/바탕 화면/LGAxSchool/핵심프로젝트/정수기프로젝트/01_데이터분석/그래프/그래프_통합상담_근거범위.png


![월별 상담 추이](그래프_13개월_상담추이.png)

*정수기대여·냉장고·TV 월별 1372 상담건수. 서로 다른 품목이며 고객 수가 아니다.*

![표준답변 문서 구성](그래프_표준답변_문서구성.png)

*정수기대여 표준답변의 원본 분류별 문서 수. 실제 상담 유형별 빈도가 아니다.*

![피해구제와 공식 FAQ의 별도 근거](그래프_통합상담_근거범위.png)

*왼쪽과 오른쪽은 서로 다른 분모·척도이므로 막대 길이나 비율을 직접 비교하지 않는다.*


## 5. 인사이트 판정

| 구분 | 이 노트북에서 확인한 것 | 말할 수 없는 것 |
|---|---|---|
| 상담 규모 | 정수기대여 13개월 합계 6,902건, 매월 접수 | 고유 고객 수, LG 고객 비중, 구매 전 문의 건수 |
| 문제 유형 | KCA 피해구제에서 계약 823건·품질/AS 471건 | 1372 일반 상담의 동일한 유형 비중 |
| 답변 자료 | 표준답변·LG FAQ·제품 상세·구독 안내·지원 문서의 조건 | FAQ 항목 수를 고객 수요·불편 강도로 일반화 |
| 제품 비교 | WD520VCT와 WU923AWB의 설치 형태·기능이 다름 | 고객에게 항상 가장 적합한 제품, 설치 가능성·가격·재고 |

**검증할 한 가지 CX 문제:** 정수기 구매·이용 고객이 제품·설치·사용·관리·계약 조건을 여러 문서에 걸쳐 확인한 뒤 자신에게 맞는 선택과 다음 행동을 판단하기 어려울 수 있다. 이는 공개 자료에서 도출한 **가설**이다. 고객 인터뷰와 기존 LG FAQ/AI Chat 대비 동일 과제 평가가 필요하다.


## 현재 서비스 범위
정수기 구독 고객은 유지·교체·해지를 고민하는 상황에서, 자신의 계약기간·종료 비용과 새 구독 조건을 함께 확인하기 어려워 다음 행동을 판단하는 데 불편을 겪는다.
계약 조회·정산·새 모델 재구독 비교·상담 준비에 집중한다. 사용방법·오류는 공식 AI Chat 연결. 기존 통계 결과는 변경하지 않으며 재구독 효과를 입증한 통계로 해석하지 않는다.